# BÁO CÁO PHẦN 2 - BÀI 2 (TRANG 37, 38): ONLINE RETAIL DATASET (UNITED KINGDOM)
## CÂU 4: Gom nhóm dữ liệu giỏ hàng nước Anh và chuyển đổi sang One-Hot Encoding
- Lọc các hóa đơn từ nước Anh (`Country == 'United Kingdom'`)
- Gom nhóm `Quantity` theo `InvoiceNo` và `Description`
- Chuyển đổi bảng sang dạng ma trận hóa đơn - mặt hàng (Unstack)
- Mã hóa sang ma trận One-Hot dạng nhị phân (0 và 1 / True và False)
- Xóa cột cước phí vận chuyển `'POSTAGE'` nếu có


In [1]:
# ==========================================
# THÔNG TIN SINH VIÊN THỰC HIỆN
# Họ và tên: Hoàng Thành Đạt
# MSSV: 6451071016
# ==========================================
ho_ten = "Hoàng Thành Đạt"
mssv = "6451071016"
print(f"Họ tên: {ho_ten} - MSSV: {mssv}")

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder

def get_data_path(rel_filename):
    for candidate in [
        os.path.join("..", "..", "dataset", rel_filename),
        os.path.join("..", "dataset", rel_filename),
        os.path.join("dataset", rel_filename),
        rel_filename
    ]:
        if os.path.exists(candidate):
            return candidate
    return rel_filename

def load_online_retail():
    for p in ['../../dataset/Online_Retail.pkl', '../dataset/Online_Retail.pkl', 'dataset/Online_Retail.pkl', 'Online_Retail.pkl']:
        if os.path.exists(p):
            return pd.read_pickle(p)
    path = get_data_path('Online Retail.xlsx')
    return pd.read_excel(path)


df = load_online_retail()
df['Description'] = df['Description'].astype(str).str.strip()
df.dropna(axis=0, subset=['InvoiceNo'], inplace=True)
df['InvoiceNo'] = df['InvoiceNo'].astype('str')
df = df[~df['InvoiceNo'].str.contains('C')]

# 1. Gom nhóm số lượng theo InvoiceNo và Description cho nước Anh
basket = (df[df['Country'] == 'United Kingdom']
          .groupby(['InvoiceNo', 'Description'])['Quantity']
          .sum().unstack().reset_index().fillna(0)
          .set_index('InvoiceNo'))

# 2. Hàm chuyển đổi hot-encoding thành one-hot encoding (0 và 1)
def encode_data(datapoint):
    if datapoint <= 0:
        return 0
    if datapoint >= 1:
        return 1

# Sử dụng map (hoặc applymap)
basket = basket.map(encode_data)

# 3. Xóa cột 'POSTAGE' nếu tồn tại
if 'POSTAGE' in basket.columns:
    basket.drop('POSTAGE', inplace=True, axis=1)

print("--- KÍCH THƯỚC MA TRẬN GIỎ HÀNG (BASKET MATRIX) NƯỚC ANH ---")
print(f"Số lượng hóa đơn (giao dịch): {basket.shape[0]}")
print(f"Số lượng mặt hàng khác nhau: {basket.shape[1]}")

print("\n--- MẪU 5 DÒNG VÀ 5 CỘT CỦA MA TRẬN ONE-HOT ---")
print(basket.iloc[:5, :5])


Họ tên: Hoàng Thành Đạt - MSSV: 6451071016
--- KÍCH THƯỚC MA TRẬN GIỎ HÀNG (BASKET MATRIX) NƯỚC ANH ---
Số lượng hóa đơn (giao dịch): 18668
Số lượng mặt hàng khác nhau: 4175

--- MẪU 5 DÒNG VÀ 5 CỘT CỦA MA TRẬN ONE-HOT ---
Description  *Boombox Ipod Classic  ...  12 DAISY PEGS IN WOOD BOX
InvoiceNo                           ...                           
536365                           0  ...                          0
536366                           0  ...                          0
536367                           0  ...                          0
536368                           0  ...                          0
536369                           0  ...                          0

[5 rows x 5 columns]


### Nhận xét Câu 4:
- Ma trận giỏ hàng của nước Anh gồm 18.668 hóa đơn và 4.175 mặt hàng phân biệt.
- Mỗi hàng đại diện cho 1 hóa đơn mua sắm duy nhất, mỗi cột là 1 mặt hàng, nhận giá trị 1 nếu hóa đơn có chứa mặt hàng đó và 0 nếu không chứa.
- Cột `POSTAGE` (chi phí bưu chính) đã được xóa bỏ để tránh việc luật kết hợp sinh ra các kết quả hiển nhiên không có giá trị phân tích kinh doanh.
